# CRIVO — recuperar o treino e limitar o armazenamento
Use este notebook **depois de concluir o pré-treino**, com o notebook antigo parado.
Ele reaproveita os seus pesos: não repete os 30.000 passos e não baixa modelos externos.

**Executar tudo**, com as opções padrão, confere os resultados e mede o armazenamento contabilizado pelo Drive, sem apagar arquivos. Versões antigas sem `keepForever` não são suficientes para explicar quota cheia. O diagnóstico usa CPU; não precisa de GPU. A limpeza opcional de revisões fica desligada e mantém os arquivos atuais, as duas versões mais novas e versões fixadas manualmente.

O piloto de diálogo está **desligado por padrão**. Quando ligado, tem no máximo 300 passos, salva no disco local e mantém dois backups verificados no Drive. Um piloto não garante boa conversação; nenhum peso é ativado no site automaticamente.


In [ ]:
from google.colab import drive, auth
from pathlib import Path
import hashlib, json, shutil, subprocess, sys

drive.mount('/content/drive')
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
credenciais, _ = google.auth.default()
servico = build('drive', 'v3', credentials=credenciais, cache_discovery=False)
BASE = '/content/drive/MyDrive/CRIVO/dialogos-amplos-v2' # @param {type:"string"}
BASE = Path(BASE)
registro = json.loads((BASE/'ultimo_experimento.json').read_text())
ORIGEM = BASE / registro['experimento']
r_pre = json.loads((ORIGEM/'pretreino/relatorio.json').read_text())
if not r_pre.get('concluido'):
    raise RuntimeError('Este notebook exige pré-treino concluído; não reinicia um treino incompleto.')
print('Pré-treino preservado:', r_pre['passo'], '/', r_pre['horizonte'])
for nome in ('dialogo/relatorio.json', 'dialogo/melhor/relatorio.json'):
    p = ORIGEM/nome
    if p.exists():
        r = json.loads(p.read_text())
        print(nome, 'passo:', r['passo'], 'parada pela validação:', r.get('parada_validacao'))

# Uma configuração separada preserva a revisão do piloto entre sessões.
PONTEIRO_RECUPERACAO = BASE/'recuperacao_dialogo_v1.json'
rec = json.loads(PONTEIRO_RECUPERACAO.read_text()) if PONTEIRO_RECUPERACAO.exists() else {}
if rec and rec['origem'] != registro['experimento']:
    raise RuntimeError('A recuperação pertence a outro experimento. Preserve-a antes de trocar de origem.')
ROOT = Path('/content/CRIVO-recuperacao-v1')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','--branch','codex/dialogos-colab-amplos',
                    'https://github.com/HROSONE/CRIVO.git',str(ROOT)],check=True)
if subprocess.run(['git','status','--porcelain'],cwd=ROOT,capture_output=True,text=True,check=True).stdout.strip():
    raise RuntimeError('Checkout alterado: preserve suas mudanças antes de continuar.')
if rec:
    subprocess.run(['git','fetch','--depth','1','origin',rec['codigo']],cwd=ROOT,check=True)
    subprocess.run(['git','checkout','--detach','FETCH_HEAD'],cwd=ROOT,check=True)
REVISAO = subprocess.run(['git','rev-parse','HEAD'],cwd=ROOT,capture_output=True,text=True,check=True).stdout.strip()
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from scripts.backup_treino_drive import (digest, empacotar, publicar, snapshots,
                                       baixar, restaurar, limpar_revisoes_antigas, _listar, _escapar)
print('Código fixado para a recuperação:', REVISAO)


## Conferir a quota real antes de apagar qualquer coisa
O relatório registra `storageQuota` da conta conectada e `quotaBytesUsed` dos binários deste experimento, inclusive se pertencem a outra conta. A quota pode incluir Gmail/Photos, lixeira e outros projetos.

As revisões comuns não fixadas não devem ser somadas como espaço ocupado: [a API contabiliza a revisão atual e as versões `keepForever`](https://developers.google.com/workspace/drive/api/reference/rest/v3/files#File.FIELDS.quota_bytes_used). Limpar o histórico por si só pode não liberar quota.

`LIMPAR_REVISOES` é opcional e vem desligado. Se ativado, remove permanentemente apenas revisões não fixadas antigas, preservando os arquivos atuais e as duas versões mais novas. Não esvazia a lixeira. Pode exigir a conta proprietária e espaço temporário para a operação da API.


In [ ]:
LIMPAR_REVISOES = False # @param {type:"boolean"}
def filhos(pai):
    return _listar(servico.files(), q="'"+_escapar(pai)+"' in parents and trashed=false",
                   pageSize=100, fields='nextPageToken,files(id,name,mimeType,parents)')
q = "name='"+_escapar(registro['experimento'])+"' and mimeType='application/vnd.google-apps.folder' and trashed=false"
candidatos = _listar(servico.files(),q=q,pageSize=100,fields='nextPageToken,files(id,name,parents)')
correspondencias=[]
for c in candidatos:
    for pai in c.get('parents',[]):
        if servico.files().get(fileId=pai,fields='name').execute()['name'] == BASE.name:
            correspondencias.append(c)
if len(correspondencias)!=1:
    raise RuntimeError('Não foi possível identificar uma única pasta do experimento; nada será removido.')
PASTA_ORIGEM_ID = correspondencias[0]['id']
alvos=[]
for etapa in filhos(PASTA_ORIGEM_ID):
    if etapa['name'] not in ('pretreino','dialogo') or etapa['mimeType']!='application/vnd.google-apps.folder':
        continue
    pastas=[etapa]
    pastas += [f for f in filhos(etapa['id']) if f['name']=='melhor' and f['mimeType']=='application/vnd.google-apps.folder']
    for pasta in pastas:
        for f in filhos(pasta['id']):
            if f['name'] in ('checkpoint.pt','pesos.pt'):
                alvos.append((f['id'],pasta['id']))
quota=servico.about().get(fields='storageQuota').execute().get('storageQuota',{})
diagnostico=dict(quota_conta_conectada=quota,arquivos=[],historico=[],
    observacao='Versões sem keepForever não entram na soma de quotaBytesUsed. Arquivos compartilhados podem pertencer a outra conta.')
print('Quota da conta conectada:',quota)
for file_id,parent_id in alvos:
    meta=servico.files().get(fileId=file_id,fields='id,name,size,quotaBytesUsed,ownedByMe,md5Checksum').execute()
    diagnostico['arquivos'].append(meta)
    print('Arquivo:',meta['name'],'| bytes atuais:',meta.get('size'),'| bytes contabilizados:',meta.get('quotaBytesUsed'),'| desta conta:',meta.get('ownedByMe'))
    resultado=limpar_revisoes_antigas(servico,file_id,parent_id,executar=LIMPAR_REVISOES)
    diagnostico['historico'].append(resultado)
    print(resultado)
(ORIGEM/'diagnostico_armazenamento.json').write_text(json.dumps(diagnostico,indent=2)+'\n')
print('Diagnóstico salvo:',ORIGEM/'diagnostico_armazenamento.json')
print('Arquivos atuais preservados. Pré-treino não foi refeito.')


## Piloto opcional: somente 300 passos de diálogo
Deixe desligado para apenas recuperar espaço e conferir o treino. Para experimentar, ative GPU e marque a opção abaixo. O piloto usa taxa de aprendizagem menor, peso igual por resposta amostrada, validação a cada 50 passos e parada antecipada. Escolhe o melhor resultado pela validação; teste reservado e avaliação humana continuam necessários.

O checkpoint do piloto inclui Adam e RNG. Ao reabrir este notebook, ele restaura o último backup íntegro e a mesma revisão do código. Uma desconexão pode perder o trabalho desde o último backup remoto, limitado a um bloco de até 100 passos. Falha no upload interrompe a continuação antes do próximo bloco.


In [ ]:
TREINAR_PILOTO = False # @param {type:"boolean"}
if TREINAR_PILOTO:
    subprocess.run([sys.executable,'-m','pip','install','-q','tokenizers>=0.20,<1','numpy>=1.24,<3','pyarrow>=15,<26'],check=True)
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Ative GPU para o piloto; diagnóstico e consulta não precisam dela.')
    LOCAL = Path('/content/crivo-recuperacao-v1')
    LOCAL.mkdir(exist_ok=True)
    INICIAL = LOCAL/'inicial'
    INICIAL.mkdir(exist_ok=True)
    fonte = ORIGEM/'pretreino/melhor'
    r_fonte = json.loads((fonte/'relatorio.json').read_text())
    for nome in ('pesos.pt','tokenizer.json'):
        shutil.copyfile(fonte/nome,INICIAL/nome)
    if digest(INICIAL/'pesos.pt') != r_fonte['pesos_sha256']:
        raise RuntimeError('Pesos copiados não correspondem ao relatório original.')
    if rec and rec['pesos_iniciais'] != r_fonte['pesos_sha256']:
        raise RuntimeError('A origem mudou. Não misturar experimentos.')
    RUN = registro['experimento']+'-recuperacao-v1'
    if not rec:
        pasta=servico.files().create(body=dict(name=RUN,mimeType='application/vnd.google-apps.folder'),fields='id').execute()
        rec=dict(origem=registro['experimento'],codigo=REVISAO,pesos_iniciais=r_fonte['pesos_sha256'],pasta_backup=pasta['id'])
        PONTEIRO_RECUPERACAO.write_text(json.dumps(rec,indent=2)+'\n')
    PASTA_BACKUP_ID = rec['pasta_backup']
    ETAPA = LOCAL/'dialogo'
    if not ETAPA.exists():
        disponiveis=snapshots(servico,PASTA_BACKUP_ID,RUN,'dialogo')
        if disponiveis:
            erros=[]
            for meta in disponiveis:
                try:
                    baixar(servico,meta,LOCAL/'restaurar.zip')
                    restaurar(LOCAL/'restaurar.zip',ETAPA)
                    print('Backup íntegro restaurado:',meta['name'])
                    break
                except Exception as e:
                    erros.append(str(e))
            else:
                raise RuntimeError('Nenhum backup pôde ser restaurado; não reiniciar do zero. '+str(erros))
    FONTES=Path('/content/crivo-fontes-dialogos-amplos')
    CORPUS=Path('/content/crivo-corpus-dialogos-amplos')
    subprocess.run([sys.executable,'-c',"from pathlib import Path; from scripts.baixar_fontes_linguagem import baixar; p=Path('/content/crivo-fontes-dialogos-amplos'); p.mkdir(exist_ok=True); baixar('oasst2',p); baixar('oasst2_completo',p); baixar('wikipedia',p)"],cwd=ROOT,check=True)
    subprocess.run([sys.executable,'scripts/preparar_conversa_gerativa.py','--oasst2',str(FONTES/'oasst2.messages.jsonl.gz'),'--oasst2-completo',str(FONTES/'oasst2.all.messages.jsonl.gz'),'--dialogos-amplos','--contexto','512','--wikipedia',str(FONTES/'wikipedia.pt.parquet'),'--saida',str(CORPUS)],cwd=ROOT,check=True)
    if digest(CORPUS/'manifesto.json') != digest(ORIGEM/'corpus_manifesto.json'):
        raise RuntimeError('Corpus diferente do original; não iniciar o piloto.')
    comum=[sys.executable,'-u','scripts/treinar_linguagem_profunda.py','--corpus',str(CORPUS),
           '--saida',str(ETAPA),'--fase','dialogo','--inicial',str(INICIAL),
           '--dimensao','384','--camadas','8','--cabecas','8','--contexto','512',
           '--passos','300','--lote','12','--lr','0.00003','--semente','20261004',
           '--threads','2','--dispositivo','cuda','--avaliar-a-cada','50','--salvar-a-cada','50',
           '--selecionar-melhor','--selecao-humana','--podar-padding','--perda-por-resposta',
           '--repeticao-linguagem','0.15','--paciencia-validacoes','4','--max-segundos','600']
    pendente=LOCAL/'backup_pendente'
    def confirmar_backup():
        empacotar(ETAPA,LOCAL/'backup.zip')
        resultado=publicar(servico,LOCAL/'backup.zip',PASTA_BACKUP_ID,RUN,'dialogo')
        if pendente.exists(): pendente.unlink()
        return resultado
    for _ in range(12):
        if pendente.exists():
            confirmar_backup()
        r=json.loads((ETAPA/'relatorio.json').read_text()) if (ETAPA/'relatorio.json').exists() else {'passo':0}
        if r.get('concluido') or r.get('parada_validacao'):
            print('Piloto encerrado:',r['passo']); break
        cmd=comum+['--parar-em',str(min(300,r['passo']+100))]
        if (ETAPA/'checkpoint.pt').exists():
            indice=cmd.index('--inicial')
            del cmd[indice:indice+2]
            cmd+=['--retomar']
        subprocess.run(cmd,cwd=ROOT,check=True)
        pendente.write_text('Upload necessário antes do próximo bloco.\n')
        resultado=confirmar_backup()
        r=json.loads((ETAPA/'relatorio.json').read_text())
        print('Passos confirmados no Drive:',r['passo'],'/ 300 | Backup:',resultado['id'])
    melhor=json.loads((ETAPA/'melhor/relatorio.json').read_text())
    print('Melhor passo:',melhor['passo'],'| Validação:',melhor['historico'][-1]['avaliacao'])
    print('Nenhum modelo foi promovido ao site. Compare respostas livres antes de decidir outro treino.')
else:
    print('Piloto desligado: nenhum passo de treino ou cota de GPU foi gasto por este notebook.')
